# PO4AO development and diagnostics
This notebook uses the installed `xapp.po4aoTrainer` numerical core. The first section runs a small, synthetic CPU example with the same models and iteration method used by the managed worker. It never connects INDI, opens instrument streams, publishes an engine or starts the companion app. Keep initialization separate from repeated iteration cells.

For operator-run live experiments, replace the explicit configuration and `ReplaySnapshot` inputs with your existing acquisition cells. Resolve configuration with `load_training(training_config, runtime_config)` and use `stable_replay_snapshot` for complete stream copies. Pair copies with the C++ collection generation/ready readbacks before and after copying. Call `session.invalidate_replay()` at recollection admission and wait for the newer complete warmup snapshot. Select the training device explicitly. The operator manages notebook/app responsibility; the GUI cannot pause or reset this notebook.

The companion must be reset when handing its session over to notebook experiments. The original personal notebook is left intact; only its numerical initialization/iteration cells need replacing with the shared calls below. Existing custom plots can still inspect `session.policy`, `session.dynamics`, both optimizers and replay arrays.

In [ ]:
import json
from pathlib import Path
import tempfile
import numpy as np
from xapp.po4aoTrainer.training import TrainingSession
from xapp.po4aoTrainer.po4ao_runtime import ReplaySnapshot
from xapp.po4aoTrainer.observer import save_diagnostics, load_diagnostics

# Deliberately small CPU demonstration, not the deployed training defaults.
config = json.loads(r'''{
  "runtime": {
    "stream_n_modes": 6,
    "n_modes_controlled": 4,
    "version": 1
  },
  "MDP": {
    "n_history": 3,
    "history_capacity": 8,
    "control_delay": 0,
    "planning_horizon": 2
  },
  "RL": {
    "episode_length": 32,
    "loss_function_penalty": 0.2
  },
  "integrator": {
    "gain": 0.3
  },
  "NN_models": {
    "training_batch": 8,
    "policy_architecture": "nonlinear_local"
  },
  "nonlinear": {
    "architecture": "nonlinear_local",
    "channels": 4,
    "shared_layers": 2,
    "negative_slope": 0.1,
    "low_order_modes": 4,
    "global_hidden": 4,
    "global_features": 2
  },
  "training": {
    "dynamics_pretrain_steps": 3,
    "dynamics_grad_steps": 2,
    "policy_grad_steps": 2,
    "dynamics_learning_rate": 0.001,
    "policy_learning_rate": 0.001,
    "dynamics_gradient_clip_norm": 0.5,
    "policy_gradient_clip_norm": 0.5,
    "validation_episodes": 1,
    "diagnostics_interval": 1,
    "model_publish_interval": 1,
    "first_model_publish_iteration": 1,
    "max_policy_clip_fraction": 0.02,
    "max_candidate_action_rms_ratio": 10.0,
    "training_iterations": 3
  },
  "normalization": {
    "rms_floor_fraction": 0.001
  },
  "replay_buffers": {
    "capacity": 96,
    "historical_seconds": 120,
    "historical_episodes": 20,
    "train_warmup_percent": 0.2,
    "train_historical_percent": 0.2
  },
  "policy_update": {
    "weight": 1.0,
    "anchor_samples": 8
  },
  "diagnostics": {
    "sample_rate_hz": 2000
  },
  "safety": {
    "max_command_abs": 100.0
  }
}''')
session = TrainingSession(config, device="cpu")
rng = np.random.default_rng(42)
replay_counter = 0
warmup_data = rng.normal(size=(6, 96, 2)).astype(np.float32)
warmup = ReplaySnapshot(warmup_data, 1, 96, 3, "0000000000000000")
run_directory = Path(tempfile.mkdtemp(prefix="po4ao-notebook-"))
print(session.session_id, run_directory)

## Repeat this cell to continue learning
This preserves the same models, Adam states, normalization and policy reference. Synthetic input below deliberately advances its counter on each call. Real acquisition must supply genuinely new complete data; never invent counters for recorded or instrument data.

In [ ]:
replay_counter += 1
recent = ReplaySnapshot(
    rng.normal(size=(6, 96, 2)).astype(np.float32), replay_counter, 96, 3,
    "0000000000000000",
)
result = session.step(recent, warmup, active_model_id="0000000000000000")
print(session.state, session.iteration)
if result is not None and result["record"] is not None:
    snapshot_path = save_diagnostics(session, run_directory)
    print(snapshot_path)

## Existing training and model-debug plots
Direct plots retain access to the models. Recorded snapshots provide the same diagnostic plots while the companion owns training, without accessing the worker's GPU or model objects.

In [ ]:
from xapp.po4aoTrainer.po4ao_diagnostics import (
    plot_training_diagnostics, plot_model_debug_diagnostics,
)
if session.metrics_history:
    training_figure = plot_training_diagnostics(
        session.metrics_history, session.obs_replay, session.act_replay,
        sample_rate_hz=config["diagnostics"]["sample_rate_hz"], policy=session.policy,
    )
    debug_figure = plot_model_debug_diagnostics(
        session.metrics_history, policy=session.policy, dynamics=session.dynamics,
    )

## Observe a managed run
Replace `diagnostics_directory` below with an explicit managed run directory such as `/data/po4aoTrainer/run-.../diagnostics`, or copy that directory to the workstation first. Select one complete snapshot; do not combine arrays/metadata from different iterations. This is file-based observation, not a resumable model/optimizer checkpoint.

In [ ]:
diagnostics_directory = run_directory  # Change to the chosen managed run's diagnostics directory.
snapshots = sorted(diagnostics_directory.glob("*.npz"), key=lambda p: p.stat().st_mtime_ns)
if snapshots:
    observed = load_diagnostics(snapshots[-1])
    print(observed.metadata["session_id"], observed.metadata["iteration"], observed.age_seconds)
    observed_training_figure, observed_debug_figure = observed.plot()

## Configuration and explicit notebook export
For live configuration, `load_training` validates the training TOML against the runtime geometry. At an iteration boundary, call `apply_hot(session, candidate)` for a valid revision with `needs_reset(session.config, candidate) == False`. Reinitialize explicitly for reset-only revisions.

The original `publish_tensorrt_engine` helper is available for operator-controlled notebook export. Export only after the current record reports `publication_status == "ready"`, using `session.policy_copy`, the matching dynamics model, resolved config and a separate candidate directory. It is synchronous; the managed worker instead runs it in a cancellable child process. After explicitly deploying/reloading the matching engine, call `session.stage_publication(model_id)` and pass the actual loaded model ID to subsequent iterations. Do not write into a managed immutable candidate directory or claim that notebook exports are eligible managed-session publications. The companion does not import notebook optimizer state or adopt its artifacts.